# R08. When the interpreter stops

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamnd/cpython-internals/blob/main/lessons/r08-when-the-interpreter-stops/r08.ipynb)

Your program reaches its last line. Nothing else of yours is going to run, and the process has not exited yet. What happens in between is a fixed sequence in one C function, and it is the part of the runtime that makes the fewest promises.

Most of the time you never notice. You notice when a log file comes out empty, when a temporary directory is still there in the morning, or when something you registered simply did not happen. All three of those are the same fact seen from different sides: the end of a program is not a normal place to be running Python, and the further into it you get, the less of Python is left.

![a pipeline from your last line through joining threads and running atexit callbacks to tearing down modules and objects](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/r08-when-the-interpreter-stops/diagrams/the-order-of-the-end.svg)

## About the source references

Now and then this lesson points at CPython's own source, like this: `Python/pylifecycle.c:2380-2419@v3.15.0rc1`.

Read it as three parts: the file, the lines, and the release those line numbers belong to. Sometimes there is a fourth part after a `#`, which is the name of the thing those lines are inside.

Every reference is a link, and every one is checked against the pinned source on each change, so a stale reference fails the build instead of sending you somewhere wrong. You never have to read any of it. The references are there so you can go deeper when you want to, and so you can check that this lesson is not making things up.

## Setup

Colab does not come with the small package these lessons use, so the next cell installs it. If you are running this from a checkout of the repository it is already installed and the cell does nothing.

In [ ]:
import sys

if sys.version_info < (3, 14):
    print("This lesson needs CPython 3.14 or newer.")
    print(f"This runtime is {sys.version.split()[0]}, and the cells below will not run on it.")
else:
    try:
        import pyxray
    except ImportError:
        %pip install -q "pyxray @ git+https://github.com/tamnd/cpython-internals@main#subdirectory=pyxray"
        import pyxray

## Which Python is this

Every cell below starts a fresh child interpreter and lets it die, because a notebook cannot watch its own shutdown. By the time this kernel starts finalising there is nobody left to print the answer.

Some runtimes cannot start a process at all. A browser tab is the usual example. Each cell checks first and says so rather than failing, so the notebook still reads through on a runtime that cannot run it.

## Which interpreter is this

In [ ]:
import pyxray

pyxray.show()

## The order of the end

[interpreter finalisation](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#interpreter-finalisation) is one function, [Python/pylifecycle.c:2380-2419@v3.15.0rc1#_Py_Finalize](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pylifecycle.c#L2380-L2419), and reading it top to bottom is the fastest way to understand shutdown. The first thing it does is the part you can still take part in: [Python/pylifecycle.c:2272-2311@v3.15.0rc1#make_pre_finalization_calls](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pylifecycle.c#L2272-L2311) waits for your threads, then runs your atexit callbacks. Everything after that is teardown.

The wait is not the operating system joining threads. It is a call into Python: [Python/pylifecycle.c:3843-3862@v3.15.0rc1#wait_for_thread_shutdown](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pylifecycle.c#L3843-L3862) calls `threading._shutdown`, which joins every non daemon thread you started. So your threads run to completion first, and only then does anything else on this list happen.

Your threads finish first, then your atexit callbacks, then the finalizers on your module globals.

In [ ]:
import os
import subprocess

PLAIN = os.environ | {"PYTHON_COLORS": "0"}

ORDER = """
import atexit, sys, threading, time


def note(label):
    print(f"  {label:38} is_finalizing={sys.is_finalizing()}")


class Late:
    def __del__(self):
        note("a finalizer on a module global")


def worker():
    time.sleep(0.2)
    note("a thread you started, finishing")


atexit.register(note, "an atexit callback, registered first")
atexit.register(note, "an atexit callback, registered second")
keeper = Late()
threading.Thread(target=worker).start()
note("your last line")
"""


def child(code, flags=()):
    """Start a fresh interpreter, run that program, and hand back everything it did."""
    return subprocess.run(
        [sys.executable, *flags, "-c", code],
        capture_output=True,
        text=True,
        timeout=120,
        env=PLAIN,
    )


def children_work():
    """Some runtimes cannot start a process at all. A browser tab is one of them."""
    try:
        child("pass")
    except Exception:
        return False
    return True


CHILDREN = children_work()
NO_CHILDREN = "  this runtime cannot start another interpreter, so there is nothing to watch"

print(NO_CHILDREN if not CHILDREN else child(ORDER).stdout, end="")

That last column is the [finalizing flag](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#finalizing-flag), which you can read as `sys.is_finalizing()`. It is `False` while your callbacks run and `True` by the time the finalizer does, because the flag is set after the pre finalization calls are over. That makes it the honest test for whether the usual rules still apply.

## The callbacks come back in reverse

An [atexit callback](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#atexit-callback) is the last hook you get while Python is still fully working. There is one detail worth knowing and one worth being careful about.

The detail: `register` does not append. It does `PyList_Insert(callbacks, 0, ...)`, [Modules/atexitmodule.c:202-213@v3.15.0rc1#PyList_Insert](https://github.com/python/cpython/blob/v3.15.0rc1/Modules/atexitmodule.c#L202-L213), so the list is walked in the reverse of the order you built it. That is deliberate, and it is the right default. If you opened a file and then a writer on top of it, you want the writer closed first.

The care: [Modules/atexitmodule.c:102-141@v3.15.0rc1#atexit_callfuncs](https://github.com/python/cpython/blob/v3.15.0rc1/Modules/atexitmodule.c#L102-L141) copies the list before it walks it, and empties the original afterwards. Anything registered from inside a callback goes on a list nobody reads again.

![three atexit register calls stacked with the last one registered drawn on top](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/r08-when-the-interpreter-stops/diagrams/atexit-is-a-stack.svg)

Callbacks run newest first, and one registered during shutdown never runs at all.

In [ ]:
ATEXIT = """
import atexit


def note(label):
    print(f"  {label}")


def registers_another():
    note("the middle callback, which registers another one from inside itself")
    atexit.register(note, "the callback registered during shutdown")


atexit.register(note, "the first callback registered")
atexit.register(registers_another)
atexit.register(note, "the last callback registered")
print(f"  {atexit._ncallbacks()} callbacks are registered and the program is over")
"""

print(NO_CHILDREN if not CHILDREN else child(ATEXIT).stdout, end="")

## What a finalizer can still reach

After the callbacks, the interpreter starts taking things apart. Modules are torn down, and then the module dict itself is emptied: [Python/pylifecycle.c:1931-1948@v3.15.0rc1#finalize_clear_modules_dict](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pylifecycle.c#L1931-L1948). Before that there is a smaller pass that removes the entries most likely to be holding something alive, including `sys.path` and `sys.meta_path`: [Python/pylifecycle.c:1683-1702@v3.15.0rc1#finalize_modules_delete_special](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pylifecycle.c#L1683-L1702).

Emptying `sys.meta_path` is what turns the import system off. There is no flag for it. The import machinery finds the list missing and raises: [Lib/importlib/_bootstrap.py:1196-1210@v3.15.0rc1#meta_path](https://github.com/python/cpython/blob/v3.15.0rc1/Lib/importlib/_bootstrap.py#L1196-L1210).

So a `__del__` that runs this late is in an odd position. Your own module's globals are fine, because they are exactly what is being freed. Everything reached through the import system is gone, including modules you imported at the top of the file.

A finalizer running during shutdown can read your module globals but cannot import anything, not even a module already imported.

In [ ]:
LATE = """
import json, sys

MESSAGE = "still readable, it is what is being taken apart"


class Late:
    def __del__(self):
        print("  sys.is_finalizing()   ", sys.is_finalizing())
        print("  a module global       ", MESSAGE)
        print("  len(sys.modules)      ", len(sys.modules))
        try:
            import json
            print("  import json           ", "worked")
        except ImportError as unhappy:
            print("  import json           ", unhappy)


keeper = Late()
print("  before shutdown, len(sys.modules) is", len(sys.modules), "and json is imported")
"""

print(NO_CHILDREN if not CHILDREN else child(LATE).stdout, end="")

> **Version note.** The count before shutdown depends on how the child was started as much as on the version. A plain 3.14 gets 58 where a plain 3.15 gets 61, and a notebook kernel passes on enough environment to move it again. What the finalizer sees is the same everywhere.

![a table of five things a finalizer might try during shutdown and what each one gives back](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/r08-when-the-interpreter-stops/diagrams/what-a-late-finalizer-can-reach.svg)

This is the shape of a whole class of bug. A `__del__` or a `weakref` callback that imports something, or calls a function that imports something, works in every test you write and fails only on the way out, where the failure is printed and thrown away.

## When something raises on the way out

Which raises the next question: what does happen to an exception at this point? There is no code left to catch it and no sensible way to report it, so both paths take the same way out. `atexit` uses `PyErr_FormatUnraisable` and carries on with the next callback. A failing `__del__` is reported the same way.

The part that catches people is the exit status.

An exception in an atexit callback or a finalizer is printed and ignored, and the process still exits with status zero.

In [ ]:
import re

ERRORS = """
import atexit


class Breaks:
    def __del__(self):
        raise RuntimeError("this finalizer raised")


def unhappy():
    raise ValueError("this atexit callback raised")


atexit.register(unhappy)
keeper = Breaks()
print("  two things are set up to fail on the way out")
"""

if not CHILDREN:
    print(NO_CHILDREN)
else:
    done = child(ERRORS)
    print(done.stdout, end="")
    print("  exit status:", done.returncode)
    for line in done.stderr.splitlines():
        if line and not line.startswith((" ", "Traceback")):
            print("  " + re.sub(r" at 0x[0-9a-f]+", "", line))

Both failures were printed and neither changed the answer the shell gets. If your cleanup runs at exit and you rely on the status code to tell you it worked, it will not.

## The thread that stops your finalizers

Now the case that is genuinely surprising. A daemon thread is one the interpreter does not wait for, so the usual advice is that it just stops wherever it is. That is true, but it is not the whole story.

`threading._shutdown` returns without waiting, the atexit callbacks run, and then teardown begins. Meanwhile the daemon thread still has a stack, and every frame on that stack holds a reference to the globals of the module its function was defined in. If that module is yours, your module dict cannot be cleared, so nothing in it is freed and none of its finalizers run.

Which means the same thread, started two ways, gives two different endings. Pass `time.sleep` and the frame belongs to the standard library. Pass a function of your own and it belongs to you.

One daemon thread running a function from your own module stops every finalizer in that module from running.

In [ ]:
DAEMON = """
import threading, time


def snooze():
    time.sleep(30)


class Late:
    def __del__(self):
        print("  the finalizer ran")


keeper = Late()
threading.Thread(TARGET, daemon=True).start()
print("  a daemon thread is running and the program is over")
"""

if not CHILDREN:
    print(NO_CHILDREN)
else:
    for what, target in [
        ("time.sleep, a function from the standard library", "target=time.sleep, args=(30,)"),
        ("snooze, a function defined in the program itself", "target=snooze"),
    ]:
        print(" ", what)
        print(child(DAEMON.replace("TARGET", target)).stdout, end="")

![the two ways to start the same daemon thread side by side, one letting finalizers run and one not](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/r08-when-the-interpreter-stops/diagrams/the-thread-that-holds-your-globals.svg)

Nothing is broken here. The thread is alive, its frame is real, and the reference it holds is a real reference. It is just that the thing keeping your objects alive is somewhere you would never think to look.

## The one collection that still happens

There is exactly one garbage collection in the middle of all this: `PyGC_Collect()`, called once between the thread cleanup and the module teardown, [Python/pylifecycle.c:2460-2485@v3.15.0rc1#PyGC_Collect](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pylifecycle.c#L2460-L2485). It is a full collection of every generation, and it is what breaks the cycles that would otherwise keep whole object graphs alive past the point where anything can free them.

You can watch it with `gc.callbacks`, which is still installed at that point.

One full collection of generation two runs during shutdown, and the finalizing flag is already set when it does.

In [ ]:
COLLECT = """
import gc, sys


def watch(phase, info):
    if phase == "stop":
        print(f"  collected generation {info['generation']}, is_finalizing={sys.is_finalizing()}")


gc.callbacks.append(watch)


class Node:
    pass


for _ in range(3):
    one, two = Node(), Node()
    one.peer, two.peer = two, one

print("  three cycles were left behind and nothing has collected them yet")
"""

print(NO_CHILDREN if not CHILDREN else child(COLLECT).stdout, end="")

## The interpreter you forgot to close

One more thing happens before any of the above: [Python/pylifecycle.c:2843-2872@v3.15.0rc1#finalize_subinterpreters](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pylifecycle.c#L2843-L2872) looks for interpreters you created and never closed, warns about each one, and finalises them for you. Each gets the full treatment, its own atexit callbacks included, before the main interpreter carries on with its own ending.

A subinterpreter you never closed is finalised for you, with a RuntimeWarning, before the main interpreter finishes.

In [ ]:
SUBS = """
import concurrent.interpreters as interpreters

kid = interpreters.create()
kid.exec("import atexit")
kid.exec("atexit.register(print, '  the second interpreter ran its own atexit callback')")
print("  a second interpreter is open and nobody closed it", flush=True)
"""

if not CHILDREN:
    print(NO_CHILDREN)
else:
    done = child(SUBS)
    print(done.stdout, end="")
    for line in done.stderr.splitlines():
        print("  " + line.strip())

The warning is worth taking seriously. Being finalised at shutdown is not the same as being closed, because by then the main interpreter is already on its way out and the order between the two is not something you control.

## Nine endings on a build from the pinned source

Everything above ran one case at a time on whatever interpreter you have. Here is the same set of hazards run together in a container, against a release build made from the pinned source. There is also `Py_FinalizeEx`, [Python/pylifecycle.c:2604-2612@v3.15.0rc1#Py_FinalizeEx](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pylifecycle.c#L2604-L2612), which is the same function with a return value, for embedders who want to know whether flushing worked.

Which of the things you registered actually run when the interpreter stops?

```python
"""Nine ways a process can end, and what each one still runs.

Shutdown is the part of the runtime with the fewest promises, and the only honest way to watch
it is from outside. So this program starts a child interpreter for each hazard and reports
what the child managed to run and what status it left behind.

Every child registers an atexit callback and keeps one object with a finalizer in a module
global. Whether those two things run is the whole question, and the answer is not always yes.

On a debug build there is one more section. `-X showrefcount` prints how many references and
how many allocated blocks were still there when the interpreter stopped, so the cost of the
one case that goes wrong can be counted rather than described.
"""

import re
import subprocess
import sys

PREAMBLE = """
import atexit, os, sys


def note(label, _w=os.write, _f=sys.is_finalizing):
    _w(2, f"{label} finalizing={_f()}\\n".encode())


class Late:
    def __del__(self, _note=note):
        _note("finalizer")


def snooze():
    import time
    time.sleep(30)


atexit.register(note, "atexit")
keeper = Late()
"""

CASES = (
    ("an ordinary exit", ""),
    ("sys.exit with a status", "sys.exit(3)"),
    ("an unhandled exception", "raise SystemError('on purpose')"),
    ("os._exit, which skips everything", "os._exit(0)"),
    ("an atexit callback that raises", "atexit.register(lambda: 1 / 0)"),
    (
        "a finalizer that raises",
        "class Angry:\n    def __del__(self):\n        1 / 0\nbad = Angry()",
    ),
    (
        "a daemon thread inside time.sleep",
        "import threading, time\n"
        "threading.Thread(target=time.sleep, args=(30,), daemon=True).start()",
    ),
    (
        "a daemon thread running your code",
        "import threading\nthreading.Thread(target=snooze, daemon=True).start()",
    ),
    ("a subinterpreter nobody closed", "import concurrent.interpreters as it\nkid = it.create()"),
)

ORDER = """
import atexit, os, sys, threading, time


def note(label, _w=os.write, _f=sys.is_finalizing):
    _w(2, f"  {label:38} finalizing={_f()}\\n".encode())


class Late:
    def __init__(self, label):
        self.label = label

    def __del__(self, _note=note):
        _note(f"finalizer of {self.label}")


def worker():
    time.sleep(0.2)
    note("a thread you started finishing")


atexit.register(note, "atexit registered first")
atexit.register(note, "atexit registered second")
keeper = Late("a module global")
threading.Thread(target=worker).start()
note("your last line")
"""

HELD = """
import atexit, os, sys


def snooze():
    import time
    time.sleep(30)


class Held:
    pass


keeper = [Held() for _ in range(5000)]
"""

SHAPES = (
    ("nothing left behind", ""),
    ("five thousand objects in a module global", HELD),
    (
        "the same, plus a daemon thread running your code",
        HELD + "import threading\nthreading.Thread(target=snooze, daemon=True).start()\n",
    ),
)

LEFTOVER = re.compile(r"\[(\d+) refs, (\d+) blocks\]")


def run(program, flags=()):
    """Start a child interpreter with that program and hand back what it did."""
    return subprocess.run(
        [sys.executable, *flags, "-c", program], capture_output=True, text=True, timeout=180
    )


def leftover(program):
    """What a debug build reports was still alive after it finished shutting down."""
    found = LEFTOVER.search(run(program, ("-X", "showrefcount")).stderr)
    return (int(found.group(1)), int(found.group(2))) if found else None


DEBUG = hasattr(sys, "gettotalrefcount")

print("version:", sys.version.split()[0])
print("debug build:", DEBUG)
print("free threaded:", hasattr(sys, "_is_gil_enabled") and not sys._is_gil_enabled())
print()

print("the order things happen in, watched from one child")
for line in run(ORDER).stderr.splitlines():
    print(line.rstrip())
print()

print("what each kind of ending still runs")
print(f"  {'the child':38} {'status':>6} {'atexit':>7} {'finalizer':>10}  warned")
ran_atexit = 0
ran_finalizer = 0
zero = 0
for label, body in CASES:
    done = run(PREAMBLE + body)
    saw_atexit = "atexit finalizing" in done.stderr
    saw_final = "finalizer finalizing" in done.stderr
    warned = "yes" if "RuntimeWarning" in done.stderr else ""
    ran_atexit += saw_atexit
    ran_finalizer += saw_final
    zero += done.returncode == 0
    yes_atexit = "yes" if saw_atexit else "no"
    yes_final = "yes" if saw_final else "no"
    columns = f"{done.returncode:>6} {yes_atexit:>7} {yes_final:>10}"
    print(f"  {label:38} {columns}  {warned}".rstrip())
print()

stranded = 0
if DEBUG:
    print("what this build says was still alive when the interpreter stopped")
    base = leftover("")
    for label, program in SHAPES:
        now = leftover(program)
        stranded = max(stranded, now[0] - base[0])
        print(f"  {label:50} {now[0] - base[0]:+8} refs {now[1] - base[1]:+8} blocks")
else:
    print("a debug build would also count what was left over, and this is not one")
print()

print(f"~ children that ran their atexit callback: {ran_atexit} of {len(CASES)}")
print(f"~ children that ran their finalizer: {ran_finalizer} of {len(CASES)}")
print(f"~ children whose exit status was zero: {zero} of {len(CASES)}")
if DEBUG:
    print(f"~ references stranded by one daemon thread: {stranded}")
```

```text
version: 3.15.0rc1
debug build: False
free threaded: False

the order things happen in, watched from one child
  your last line                         finalizing=False
  a thread you started finishing         finalizing=False
  atexit registered second               finalizing=False
  atexit registered first                finalizing=False
  finalizer of a module global           finalizing=True

what each kind of ending still runs
  the child                              status  atexit  finalizer  warned
  an ordinary exit                            0     yes        yes
  sys.exit with a status                      3     yes        yes
  an unhandled exception                      1     yes        yes
  os._exit, which skips everything            0      no         no
  an atexit callback that raises              0     yes        yes
  a finalizer that raises                     0     yes        yes
  a daemon thread inside time.sleep           0     yes        yes
  a daemon thread running your code           0     yes         no
  a subinterpreter nobody closed              0     yes        yes  yes

a debug build would also count what was left over, and this is not one

~ children that ran their atexit callback: 8 of 9
~ children that ran their finalizer: 7 of 9
~ children whose exit status was zero: 7 of 9
```

That ran on Python 3.15.0rc1 in the release build this project publishes, which is `ghcr.io/tamnd/cpython-internals/cpython:release@sha256:fb55d6afcf053c974de6447fafbd2be6af20cdb9f596e25a0445607b8af981e3`. You do not need that build to read the numbers, and you do need it to produce them, which is why this is recorded rather than left as a cell you run. If you want to watch it happen yourself, `docker run --rm -i ghcr.io/tamnd/cpython-internals/cpython:release@sha256:fb55d6afcf053c974de6447fafbd2be6af20cdb9f596e25a0445607b8af981e3 python3 -` takes the program on standard input.

![a table of nine ways a process can end with whether atexit, the finalizer and the exit status came out as expected](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/r08-when-the-interpreter-stops/diagrams/what-each-ending-runs.svg)

Eight of nine ran their atexit callback and seven ran their finalizer. The two gaps are `os._exit`, which you asked for, and the daemon thread, which you did not.

Now the same program on a debug build, which counts what was still alive when the interpreter stopped.

How much does one daemon thread leave stranded when the interpreter stops?

```python
"""Nine ways a process can end, and what each one still runs.

Shutdown is the part of the runtime with the fewest promises, and the only honest way to watch
it is from outside. So this program starts a child interpreter for each hazard and reports
what the child managed to run and what status it left behind.

Every child registers an atexit callback and keeps one object with a finalizer in a module
global. Whether those two things run is the whole question, and the answer is not always yes.

On a debug build there is one more section. `-X showrefcount` prints how many references and
how many allocated blocks were still there when the interpreter stopped, so the cost of the
one case that goes wrong can be counted rather than described.
"""

import re
import subprocess
import sys

PREAMBLE = """
import atexit, os, sys


def note(label, _w=os.write, _f=sys.is_finalizing):
    _w(2, f"{label} finalizing={_f()}\\n".encode())


class Late:
    def __del__(self, _note=note):
        _note("finalizer")


def snooze():
    import time
    time.sleep(30)


atexit.register(note, "atexit")
keeper = Late()
"""

CASES = (
    ("an ordinary exit", ""),
    ("sys.exit with a status", "sys.exit(3)"),
    ("an unhandled exception", "raise SystemError('on purpose')"),
    ("os._exit, which skips everything", "os._exit(0)"),
    ("an atexit callback that raises", "atexit.register(lambda: 1 / 0)"),
    (
        "a finalizer that raises",
        "class Angry:\n    def __del__(self):\n        1 / 0\nbad = Angry()",
    ),
    (
        "a daemon thread inside time.sleep",
        "import threading, time\n"
        "threading.Thread(target=time.sleep, args=(30,), daemon=True).start()",
    ),
    (
        "a daemon thread running your code",
        "import threading\nthreading.Thread(target=snooze, daemon=True).start()",
    ),
    ("a subinterpreter nobody closed", "import concurrent.interpreters as it\nkid = it.create()"),
)

ORDER = """
import atexit, os, sys, threading, time


def note(label, _w=os.write, _f=sys.is_finalizing):
    _w(2, f"  {label:38} finalizing={_f()}\\n".encode())


class Late:
    def __init__(self, label):
        self.label = label

    def __del__(self, _note=note):
        _note(f"finalizer of {self.label}")


def worker():
    time.sleep(0.2)
    note("a thread you started finishing")


atexit.register(note, "atexit registered first")
atexit.register(note, "atexit registered second")
keeper = Late("a module global")
threading.Thread(target=worker).start()
note("your last line")
"""

HELD = """
import atexit, os, sys


def snooze():
    import time
    time.sleep(30)


class Held:
    pass


keeper = [Held() for _ in range(5000)]
"""

SHAPES = (
    ("nothing left behind", ""),
    ("five thousand objects in a module global", HELD),
    (
        "the same, plus a daemon thread running your code",
        HELD + "import threading\nthreading.Thread(target=snooze, daemon=True).start()\n",
    ),
)

LEFTOVER = re.compile(r"\[(\d+) refs, (\d+) blocks\]")


def run(program, flags=()):
    """Start a child interpreter with that program and hand back what it did."""
    return subprocess.run(
        [sys.executable, *flags, "-c", program], capture_output=True, text=True, timeout=180
    )


def leftover(program):
    """What a debug build reports was still alive after it finished shutting down."""
    found = LEFTOVER.search(run(program, ("-X", "showrefcount")).stderr)
    return (int(found.group(1)), int(found.group(2))) if found else None


DEBUG = hasattr(sys, "gettotalrefcount")

print("version:", sys.version.split()[0])
print("debug build:", DEBUG)
print("free threaded:", hasattr(sys, "_is_gil_enabled") and not sys._is_gil_enabled())
print()

print("the order things happen in, watched from one child")
for line in run(ORDER).stderr.splitlines():
    print(line.rstrip())
print()

print("what each kind of ending still runs")
print(f"  {'the child':38} {'status':>6} {'atexit':>7} {'finalizer':>10}  warned")
ran_atexit = 0
ran_finalizer = 0
zero = 0
for label, body in CASES:
    done = run(PREAMBLE + body)
    saw_atexit = "atexit finalizing" in done.stderr
    saw_final = "finalizer finalizing" in done.stderr
    warned = "yes" if "RuntimeWarning" in done.stderr else ""
    ran_atexit += saw_atexit
    ran_finalizer += saw_final
    zero += done.returncode == 0
    yes_atexit = "yes" if saw_atexit else "no"
    yes_final = "yes" if saw_final else "no"
    columns = f"{done.returncode:>6} {yes_atexit:>7} {yes_final:>10}"
    print(f"  {label:38} {columns}  {warned}".rstrip())
print()

stranded = 0
if DEBUG:
    print("what this build says was still alive when the interpreter stopped")
    base = leftover("")
    for label, program in SHAPES:
        now = leftover(program)
        stranded = max(stranded, now[0] - base[0])
        print(f"  {label:50} {now[0] - base[0]:+8} refs {now[1] - base[1]:+8} blocks")
else:
    print("a debug build would also count what was left over, and this is not one")
print()

print(f"~ children that ran their atexit callback: {ran_atexit} of {len(CASES)}")
print(f"~ children that ran their finalizer: {ran_finalizer} of {len(CASES)}")
print(f"~ children whose exit status was zero: {zero} of {len(CASES)}")
if DEBUG:
    print(f"~ references stranded by one daemon thread: {stranded}")
```

```text
version: 3.15.0rc1
debug build: True
free threaded: False

the order things happen in, watched from one child
  your last line                         finalizing=False
  a thread you started finishing         finalizing=False
  atexit registered second               finalizing=False
  atexit registered first                finalizing=False
  finalizer of a module global           finalizing=True

what each kind of ending still runs
  the child                              status  atexit  finalizer  warned
  an ordinary exit                            0     yes        yes
  sys.exit with a status                      3     yes        yes
  an unhandled exception                      1     yes        yes
  os._exit, which skips everything            0      no         no
  an atexit callback that raises              0     yes        yes
  a finalizer that raises                     0     yes        yes
  a daemon thread inside time.sleep           0     yes        yes
  a daemon thread running your code           0     yes         no
  a subinterpreter nobody closed              0     yes        yes  yes

what this build says was still alive when the interpreter stopped
  nothing left behind                                      +0 refs       +0 blocks
  five thousand objects in a module global                 +0 refs       +0 blocks
  the same, plus a daemon thread running your code     +12680 refs    +6506 blocks

~ children that ran their atexit callback: 8 of 9
~ children that ran their finalizer: 7 of 9
~ children whose exit status was zero: 7 of 9
~ references stranded by one daemon thread: 12680
```

That ran on Python 3.15.0rc1 in the debug build this project publishes, which is `ghcr.io/tamnd/cpython-internals/cpython:debug@sha256:7baea8f3dd4de2e4c3b020543729b147e636494ae9758dabffb4675793e37170`. You do not need that build to read the numbers, and you do need it to produce them, which is why this is recorded rather than left as a cell you run. If you want to watch it happen yourself, `docker run --rm -i ghcr.io/tamnd/cpython-internals/cpython:debug@sha256:7baea8f3dd4de2e4c3b020543729b147e636494ae9758dabffb4675793e37170 python3 -` takes the program on standard input.

![a bar chart of references left over for three programs, two at zero and the daemon thread one at 12680](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/r08-when-the-interpreter-stops/diagrams/what-was-left-behind.svg)

Holding five thousand objects in a module global costs nothing at the end, because the module gets cleared and they all go. Holding the same five thousand behind a running daemon thread leaves every one of them alive, along with everything the module dict reaches. That is the difference between an ending and a program that simply stopped.

## Try it yourself

**One.** Give the daemon thread a function defined in a second module of your own and import it. Whose finalizers stop running, yours or the other module's? Work out from that which module dict the frame is actually holding.

**Two.** Register an atexit callback that calls `sys.exit(1)`. Read `atexit_callfuncs` and predict what happens before you run it.

**Three.** Put a `weakref.finalize` on an object next to a `__del__` on another and see which of the two runs first. Then move one of them into a class defined in the standard library and try again.

**Four.** Run the late finalizer cell under `-X importtime`. The import that fails still costs something. Find out what.

## What you now know

Shutdown is one function read top to bottom. The first thing it does is join your non daemon threads by calling into `threading._shutdown`, then run your atexit callbacks. Everything after that is teardown, and the finalizing flag goes up in between, which is why `sys.is_finalizing()` is `False` in a callback and `True` in a late `__del__`.

Atexit callbacks run newest first, because `register` inserts at the front of a list. The list is copied and then emptied, so registering a callback from inside one is a quiet no op.

Once teardown starts, `sys.meta_path` is cleared and the import system stops working. A finalizer can still read the globals of its own module, because those are what is being freed, but any import raises `ImportError` and says why. Exceptions from both callbacks and finalizers are printed and ignored, and the exit status stays zero, so a shell script cannot tell that your cleanup failed.

Two endings skip work you asked for. `os._exit` skips all of it by design. A daemon thread running a function from your own module skips your finalizers by accident, because its stack frame holds your module's globals and stops the module dict being cleared. On a debug build that one costs 12680 references left alive, against zero for the same objects held any other way.

## What is next

R09 is the last of the runtime lessons, and it turns everything here around. Instead of watching from Python while the interpreter takes itself apart, it writes the C that has to survive it: reference counting on error paths, `tp_traverse` and `tp_clear` and `tp_finalize` on a container type, and what all of that has to look like on a build with no global interpreter lock.